# BNPL and Trade Credit Underwriting using Relational Supply-Chain Graphs with KumoRFM

<a target="_blank" href="https://colab.research.google.com/github/AbhinavKhareTech/kumo-rfm/blob/contrib/bnpl-trade-credit-notebook/notebooks/bnpl_trade_credit_underwriting.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Series:** Advanced Relational Fraud & Risk Detection with KumoRFM (Module 5 of N)
- Module 1: [Merchant Collusion Ring Detection](bfsi_fraud_detection.ipynb)
- Module 2: [Account Takeover Fraud Detection](ato_fraud_detection.ipynb)
- Module 3: [Crypto Money Laundering Detection](crypto_money_laundering_detection.ipynb)
- Module 4: Synthetic Identity Fraud Detection (coming soon)
- **Module 5: BNPL & Trade Credit Underwriting** (this notebook)

---

## Overview

BNPL and trade credit underwriting is fundamentally a **relational problem**. Default risk propagates through supply chains: when a buyer defaults on invoices to one supplier, it signals elevated risk across all their supplier relationships. Conversely, suppliers with high concentrations of risky buyers face portfolio-level distress that can cascade into their own creditworthiness.

Traditional flat-feature underwriting models score each invoice or buyer independently, missing the **network-level contagion** that drives correlated losses in trade credit portfolios.

### Why flat models fail here

In trade credit networks, default risk is driven by:
- **Buyer payment deterioration**: Gradual slowdown in payment velocity before bust-out
- **Supplier concentration risk**: A supplier dependent on a few risky buyers faces correlated losses
- **Cross-entity contagion**: One buyer's default cascades to suppliers, who then struggle to pay their own creditors
- **Network topology**: Buyers connected to many distressed suppliers carry hidden risk invisible to per-application models

The signal is **structural**: it lives in multi-hop buyer-supplier-invoice relationships, not in any single invoice's features.

### What this notebook covers

1. Synthetic multi-table dataset generation with embedded default cascades and concentration risk
2. Relational graph construction using `LocalGraph.from_data()` with temporal invoice and payment data
3. PQL-based invoice-level default scoring and buyer-level risk assessment
4. Head-to-head comparison against XGBoost with hand-engineered trade credit features
5. Supply-chain risk graph visualization showing default contagion paths
6. Portfolio-level analysis: approval rate vs expected loss tradeoff curves
7. Production deployment considerations for embedded finance platforms

### Requirements

- Python 3.9+
- A valid [KumoRFM API key](https://kumorfm.ai) (free tier available)
- Libraries: `kumoai`, `pandas`, `numpy`, `scikit-learn`, `xgboost`, `networkx`, `matplotlib`, `seaborn`

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q kumoai xgboost networkx matplotlib seaborn scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Environment ready.")

## 2. Synthetic Dataset Generation

We generate a realistic multi-table dataset modeling a B2B trade credit / BNPL platform. The schema has five interconnected tables:

| Table | Role | Key Relationships |
|---|---|---|
| `buyers` | Businesses or consumers purchasing on credit | Primary entity for risk scoring |
| `suppliers` | Merchants or vendors extending credit terms | Counterparty risk entity |
| `invoices` | Individual credit obligations (net-30, net-60, BNPL installments) | Links buyers to suppliers, temporal |
| `payments` | Partial or full payments against invoices | Temporal payment behavior |
| `credit_lines` | Active credit limits per buyer-supplier pair | Exposure tracking |

### Embedded Risk Patterns

We inject three realistic default scenarios:

1. **Gradual Payment Deterioration**: Buyers slow their payment velocity over weeks before eventually defaulting. Days-past-due increases progressively.
2. **Supplier Concentration Cascade**: A cluster of buyers connected to the same high-risk supplier default in a correlated wave.
3. **Bust-Out Pattern**: Buyers aggressively draw down credit across multiple suppliers simultaneously, then disappear.

Each pattern creates **structural graph anomalies** that flat per-invoice features cannot fully capture.

In [ ]:
# ==============================================================
# 2.1 Generate entity tables
# ==============================================================

N_BUYERS = 400
N_SUPPLIERS = 60

buyer_ids = [f"B{i:04d}" for i in range(N_BUYERS)]
industries = ["retail", "manufacturing", "services", "logistics",
              "food_bev", "construction", "tech", "healthcare"]

buyers_df = pd.DataFrame({
    "buyer_id": buyer_ids,
    "onboarding_date": pd.date_range("2023-01-01", periods=N_BUYERS, freq="6h"),
    "industry": np.random.choice(industries, N_BUYERS),
    "buyer_size": np.random.choice(["micro", "small", "medium"],
                                    N_BUYERS, p=[0.45, 0.35, 0.20]),
    "years_in_business": np.random.randint(1, 25, N_BUYERS),
    "bureau_score": np.random.normal(650, 80, N_BUYERS).clip(300, 850).astype(int),
    "is_defaulter": 0,
})

supplier_ids = [f"S{i:03d}" for i in range(N_SUPPLIERS)]
supplier_categories = ["raw_materials", "finished_goods", "services",
                       "equipment", "consumables", "logistics"]

suppliers_df = pd.DataFrame({
    "supplier_id": supplier_ids,
    "registered_date": pd.date_range("2022-01-01", periods=N_SUPPLIERS, freq="5D"),
    "category": np.random.choice(supplier_categories, N_SUPPLIERS),
    "avg_invoice_value": np.random.lognormal(mean=9.5, sigma=0.8,
                                              size=N_SUPPLIERS).clip(5000, 500000).round(0),
    "supplier_rating": np.random.choice(["A", "B", "C", "D"],
                                         N_SUPPLIERS, p=[0.3, 0.35, 0.25, 0.1]),
})

print(f"Buyers: {len(buyers_df)}, Suppliers: {len(suppliers_df)}")
buyers_df.head()

In [ ]:
# ==============================================================
# 2.2 Generate buyer-supplier relationships and credit lines
# ==============================================================

buyer_supplier_pairs = []
for bid in buyer_ids:
    n_suppliers = np.random.choice([2, 3, 4, 5, 6], p=[0.15, 0.3, 0.3, 0.15, 0.1])
    chosen = np.random.choice(supplier_ids, n_suppliers, replace=False)
    for sid in chosen:
        buyer_supplier_pairs.append({"buyer_id": bid, "supplier_id": sid})

bs_pairs_df = pd.DataFrame(buyer_supplier_pairs)

credit_lines_df = bs_pairs_df.copy()
credit_lines_df["credit_line_id"] = [f"CL{i:05d}" for i in range(len(credit_lines_df))]
credit_lines_df["credit_limit"] = np.random.choice(
    [25000, 50000, 100000, 200000, 500000],
    len(credit_lines_df), p=[0.25, 0.3, 0.25, 0.15, 0.05]
)
credit_lines_df["terms_days"] = np.random.choice([15, 30, 45, 60, 90],
                                                   len(credit_lines_df),
                                                   p=[0.1, 0.4, 0.2, 0.2, 0.1])
credit_lines_df["opened_date"] = pd.date_range(
    "2023-03-01", periods=len(credit_lines_df), freq="20min"
)

print(f"Buyer-supplier pairs: {len(bs_pairs_df)}")
print(f"Credit lines: {len(credit_lines_df)}")
print(f"Avg suppliers per buyer: {bs_pairs_df.groupby('buyer_id').size().mean():.1f}")

In [ ]:
# ==============================================================
# 2.3 Generate legitimate invoices
# ==============================================================

BASE_DATE = pd.Timestamp("2023-06-01")
N_LEGIT_INVOICES = 8000

invoice_records = []
for i in range(N_LEGIT_INVOICES):
    pair = bs_pairs_df.sample(1).iloc[0]
    bid, sid = pair["buyer_id"], pair["supplier_id"]
    avg_val = suppliers_df.loc[
        suppliers_df["supplier_id"] == sid, "avg_invoice_value"
    ].iloc[0]

    invoice_date = BASE_DATE + pd.Timedelta(days=np.random.randint(0, 180))
    amount = np.random.lognormal(
        mean=np.log(avg_val), sigma=0.4
    ).clip(1000, 1000000)

    terms = credit_lines_df.loc[
        (credit_lines_df["buyer_id"] == bid) &
        (credit_lines_df["supplier_id"] == sid),
        "terms_days"
    ]
    term_days = terms.iloc[0] if len(terms) > 0 else 30
    due_date = invoice_date + pd.Timedelta(days=int(term_days))

    invoice_records.append({
        "invoice_id": f"INV{i:06d}",
        "buyer_id": bid,
        "supplier_id": sid,
        "invoice_date": invoice_date,
        "due_date": due_date,
        "amount": round(amount, 2),
        "terms_days": term_days,
        "is_defaulted": False,
    })

invoices_df = pd.DataFrame(invoice_records)
print(f"Legitimate invoices: {len(invoices_df)}")

In [ ]:
# ==============================================================
# 2.4 Generate payments for legitimate invoices
# ==============================================================

payment_records = []
pid_counter = 0

for _, inv in invoices_df.iterrows():
    days_to_pay = int(inv["terms_days"] * np.random.uniform(0.5, 1.3))
    pay_date = inv["invoice_date"] + pd.Timedelta(days=days_to_pay)

    n_payments = np.random.choice([1, 2, 3], p=[0.7, 0.2, 0.1])
    remaining = inv["amount"]

    for j in range(n_payments):
        if j == n_payments - 1:
            pay_amount = remaining
        else:
            pay_amount = round(remaining * np.random.uniform(0.3, 0.6), 2)
            remaining -= pay_amount

        payment_records.append({
            "payment_id": f"PAY{pid_counter:07d}",
            "invoice_id": inv["invoice_id"],
            "buyer_id": inv["buyer_id"],
            "payment_date": pay_date + pd.Timedelta(days=j * np.random.randint(3, 15)),
            "payment_amount": round(pay_amount, 2),
        })
        pid_counter += 1

payments_df = pd.DataFrame(payment_records)
print(f"Legitimate payments: {len(payments_df)}")

### 2.5 Injecting Default Risk Patterns

We now inject three distinct default scenarios into the dataset. Each creates **structural anomalies** in the buyer-supplier graph that flat per-invoice features cannot fully capture.

In [ ]:
# ==============================================================
# 2.5 Inject default risk patterns
# ==============================================================

ATO_START = pd.Timestamp("2023-10-01")
inv_counter = N_LEGIT_INVOICES
pay_counter = pid_counter

default_buyers = set()
default_invoices = []
default_payments = []

# --- Pattern 1: Gradual Payment Deterioration (20 buyers) ---
deterioration_buyers = list(np.random.choice(
    [b for b in buyer_ids if b not in default_buyers], 20, replace=False
))
default_buyers.update(deterioration_buyers)

for bid in deterioration_buyers:
    buyer_sups = bs_pairs_df[bs_pairs_df["buyer_id"] == bid]["supplier_id"].tolist()
    if not buyer_sups:
        continue

    for week in range(8):
        sid = np.random.choice(buyer_sups)
        inv_date = ATO_START + pd.Timedelta(weeks=week)
        amount = np.random.uniform(10000, 80000)
        terms = 30
        is_default = week >= 6

        default_invoices.append({
            "invoice_id": f"INV{inv_counter:06d}",
            "buyer_id": bid,
            "supplier_id": sid,
            "invoice_date": inv_date,
            "due_date": inv_date + pd.Timedelta(days=terms),
            "amount": round(amount, 2),
            "terms_days": terms,
            "is_defaulted": is_default,
        })

        if not is_default:
            dpd = int(5 + week * 8 + np.random.randint(0, 5))
            default_payments.append({
                "payment_id": f"PAY{pay_counter:07d}",
                "invoice_id": f"INV{inv_counter:06d}",
                "buyer_id": bid,
                "payment_date": inv_date + pd.Timedelta(days=terms + dpd),
                "payment_amount": round(amount * np.random.uniform(0.6, 0.9), 2),
            })
            pay_counter += 1

        inv_counter += 1

# --- Pattern 2: Supplier Concentration Cascade (15 buyers) ---
risky_supplier = np.random.choice(supplier_ids)
concentration_buyers_pool = bs_pairs_df[
    bs_pairs_df["supplier_id"] == risky_supplier
]["buyer_id"].unique()
concentration_buyers = list(np.random.choice(
    [b for b in concentration_buyers_pool if b not in default_buyers],
    min(15, len([b for b in concentration_buyers_pool if b not in default_buyers])),
    replace=False
))
default_buyers.update(concentration_buyers)

for bid in concentration_buyers:
    for j in range(np.random.randint(2, 5)):
        inv_date = ATO_START + pd.Timedelta(days=np.random.randint(0, 40))
        amount = np.random.uniform(20000, 120000)
        default_invoices.append({
            "invoice_id": f"INV{inv_counter:06d}",
            "buyer_id": bid,
            "supplier_id": risky_supplier,
            "invoice_date": inv_date,
            "due_date": inv_date + pd.Timedelta(days=30),
            "amount": round(amount, 2),
            "terms_days": 30,
            "is_defaulted": True,
        })
        inv_counter += 1

# --- Pattern 3: Bust-Out (10 buyers) ---
bustout_buyers = list(np.random.choice(
    [b for b in buyer_ids if b not in default_buyers], 10, replace=False
))
default_buyers.update(bustout_buyers)

for bid in bustout_buyers:
    buyer_sups = bs_pairs_df[bs_pairs_df["buyer_id"] == bid]["supplier_id"].tolist()
    burst_start = ATO_START + pd.Timedelta(days=np.random.randint(10, 35))

    for sid in buyer_sups:
        for j in range(np.random.randint(2, 4)):
            inv_date = burst_start + pd.Timedelta(days=np.random.randint(0, 7))
            amount = np.random.uniform(50000, 200000)
            default_invoices.append({
                "invoice_id": f"INV{inv_counter:06d}",
                "buyer_id": bid,
                "supplier_id": sid,
                "invoice_date": inv_date,
                "due_date": inv_date + pd.Timedelta(days=15),
                "amount": round(amount, 2),
                "terms_days": 15,
                "is_defaulted": True,
            })
            inv_counter += 1

invoices_df = pd.concat([invoices_df, pd.DataFrame(default_invoices)], ignore_index=True)
payments_df = pd.concat([payments_df, pd.DataFrame(default_payments)], ignore_index=True)
buyers_df.loc[buyers_df["buyer_id"].isin(default_buyers), "is_defaulter"] = 1

print(f"Total invoices: {len(invoices_df)} "
      f"(defaulted: {invoices_df['is_defaulted'].sum()})")
print(f"Total payments: {len(payments_df)}")
print(f"Default buyers: {len(default_buyers)} / {N_BUYERS} "
      f"({len(default_buyers)/N_BUYERS*100:.1f}%)")
print(f"  Deterioration: {len(deterioration_buyers)}")
print(f"  Concentration cascade: {len(concentration_buyers)} "
      f"(risky supplier: {risky_supplier})")
print(f"  Bust-out: {len(bustout_buyers)}")

## 3. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

legit_inv = invoices_df[~invoices_df["is_defaulted"]]
default_inv = invoices_df[invoices_df["is_defaulted"]]

axes[0].hist(legit_inv["amount"].clip(upper=200000), bins=50, alpha=0.6,
             label="Performing", color="steelblue", density=True)
axes[0].hist(default_inv["amount"].clip(upper=200000), bins=30, alpha=0.6,
             label="Defaulted", color="crimson", density=True)
axes[0].set_title("Invoice Amount Distribution")
axes[0].set_xlabel("Amount")
axes[0].legend()

sups_per_buyer = bs_pairs_df.groupby("buyer_id").size()
default_spb = sups_per_buyer[sups_per_buyer.index.isin(default_buyers)]
legit_spb = sups_per_buyer[~sups_per_buyer.index.isin(default_buyers)]

axes[1].bar(["Performing", "Defaulters"],
            [legit_spb.mean(), default_spb.mean()],
            color=["steelblue", "crimson"], alpha=0.7)
axes[1].set_title("Avg Suppliers per Buyer")
axes[1].set_ylabel("Unique Suppliers")

pay_merged = payments_df.merge(
    invoices_df[["invoice_id", "due_date", "is_defaulted"]],
    on="invoice_id", how="left"
)
pay_merged["days_from_due"] = (
    pay_merged["payment_date"] - pay_merged["due_date"]
).dt.days

legit_pay = pay_merged[~pay_merged["is_defaulted"]]
default_pay = pay_merged[pay_merged["is_defaulted"]]

axes[2].hist(legit_pay["days_from_due"].clip(-30, 60), bins=40, alpha=0.6,
             label="Performing", color="steelblue", density=True)
if len(default_pay) > 0:
    axes[2].hist(default_pay["days_from_due"].clip(-30, 60), bins=20, alpha=0.6,
                 label="Defaulted", color="crimson", density=True)
axes[2].axvline(x=0, color="gray", linestyle="--", alpha=0.5, label="Due date")
axes[2].set_title("Payment Timing vs Due Date")
axes[2].set_xlabel("Days from Due Date")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Relational Graph Construction with KumoRFM

We construct a five-table relational graph preserving all buyer-supplier-invoice-payment relationships:

```
buyers ---+--- invoices ---+--- payments
          |                |
          +--- credit_lines
          |
suppliers-+
```

KumoRFM reasons over multi-hop paths: a buyer's risk is informed not just by their own invoices, but by the payment behavior of *other buyers* connected to the same suppliers (and vice versa). This is the structural signal flat models miss.

In [ ]:
import kumoai.experimental.rfm as rfm
import os

os.environ["KUMO_API_KEY"] = os.environ.get("KUMO_API_KEY", "ENTER_YOUR_API_KEY_HERE")
rfm.init()

In [ ]:
# ==============================================================
# 4.1 Prepare DataFrames for graph construction
# ==============================================================

buyers_graph_df = buyers_df[["buyer_id", "onboarding_date", "industry",
                              "buyer_size", "years_in_business",
                              "bureau_score", "is_defaulter"]].copy()

test_buyer_ids = set(default_buyers) | set(np.random.choice(
    [b for b in buyer_ids if b not in default_buyers],
    size=80, replace=False
))

ground_truth = buyers_graph_df[
    buyers_graph_df["buyer_id"].isin(test_buyer_ids)
][["buyer_id", "is_defaulter"]].copy()

buyers_graph_df.loc[
    buyers_graph_df["buyer_id"].isin(test_buyer_ids), "is_defaulter"
] = pd.NA

invoices_graph_df = invoices_df[["invoice_id", "buyer_id", "supplier_id",
                                  "invoice_date", "due_date", "amount",
                                  "terms_days"]].copy()
payments_graph_df = payments_df[["payment_id", "invoice_id", "buyer_id",
                                  "payment_date", "payment_amount"]].copy()
suppliers_graph_df = suppliers_df[["supplier_id", "registered_date",
                                    "category", "avg_invoice_value",
                                    "supplier_rating"]].copy()
credit_lines_graph_df = credit_lines_df[["credit_line_id", "buyer_id",
                                          "supplier_id", "credit_limit",
                                          "terms_days", "opened_date"]].copy()

print(f"Test buyers: {len(test_buyer_ids)} "
      f"(defaulters: {ground_truth['is_defaulter'].sum()}, "
      f"clean: {(ground_truth['is_defaulter']==0).sum()})")

In [ ]:
# ==============================================================
# 4.2 Build the relational graph
# ==============================================================

graph = rfm.LocalGraph.from_data({
    "buyers": buyers_graph_df,
    "suppliers": suppliers_graph_df,
    "invoices": invoices_graph_df,
    "payments": payments_graph_df,
    "credit_lines": credit_lines_graph_df,
})

print("Graph constructed. Reviewing metadata...")

In [ ]:
# ==============================================================
# 4.3 Verify and adjust metadata
# ==============================================================

graph["buyers"].time_col = "onboarding_date"
graph["suppliers"].time_col = "registered_date"
graph["invoices"].time_col = "invoice_date"
graph["payments"].time_col = "payment_date"
graph["credit_lines"].time_col = "opened_date"

graph["buyers"]["buyer_id"].stype = "ID"
graph["suppliers"]["supplier_id"].stype = "ID"
graph["invoices"]["invoice_id"].stype = "ID"
graph["payments"]["payment_id"].stype = "ID"
graph["credit_lines"]["credit_line_id"].stype = "ID"

for col in ["amount", "terms_days"]:
    graph["invoices"][col].stype = "numerical"
graph["payments"]["payment_amount"].stype = "numerical"
for col in ["credit_limit", "terms_days"]:
    graph["credit_lines"][col].stype = "numerical"
for col in ["years_in_business", "bureau_score", "is_defaulter"]:
    graph["buyers"][col].stype = "numerical"
graph["suppliers"]["avg_invoice_value"].stype = "numerical"

graph["buyers"]["industry"].stype = "categorical"
graph["buyers"]["buyer_size"].stype = "categorical"
graph["suppliers"]["category"].stype = "categorical"
graph["suppliers"]["supplier_rating"].stype = "categorical"

edge_specs = [
    ("invoices.buyer_id", "buyers.buyer_id"),
    ("invoices.supplier_id", "suppliers.supplier_id"),
    ("payments.invoice_id", "invoices.invoice_id"),
    ("payments.buyer_id", "buyers.buyer_id"),
    ("credit_lines.buyer_id", "buyers.buyer_id"),
    ("credit_lines.supplier_id", "suppliers.supplier_id"),
]
for src, tgt in edge_specs:
    try:
        graph.add_edge(src, tgt)
    except Exception:
        pass

print("Metadata verified.")
print(graph)

## 5. Default Risk Scoring with KumoRFM

We predict `is_defaulter` for test buyers using missing-value imputation. KumoRFM reasons over the full relational graph: a buyer's risk is informed by their invoice history, payment behavior, supplier connectivity, credit utilization, and crucially, the behavior of *other buyers* connected to the same suppliers.

**Note:** The cells below require a valid KumoRFM API key.

In [ ]:
# ==============================================================
# 5.1 Run KumoRFM predictions
# ==============================================================

model = rfm.KumoRFM(graph)

test_buyer_list = sorted(list(test_buyer_ids))
BATCH_SIZE = 50
kumo_predictions = []

print(f"Predicting for {len(test_buyer_list)} test buyers...")

for batch_start in range(0, len(test_buyer_list), BATCH_SIZE):
    batch = test_buyer_list[batch_start:batch_start + BATCH_SIZE]
    id_tuple = ", ".join([f"'{b}'" for b in batch])
    query = f"PREDICT buyers.is_defaulter = 1 FOR buyers.buyer_id IN ({id_tuple})"

    try:
        result = model.predict(query)
        kumo_predictions.append(result)
        if (batch_start // BATCH_SIZE) % 5 == 0:
            print(f"  Batch {batch_start//BATCH_SIZE + 1}: "
                  f"predicted {len(result)} buyers")
    except Exception as e:
        print(f"  Batch {batch_start//BATCH_SIZE + 1} error: {e}")

if kumo_predictions:
    kumo_results_df = pd.concat(kumo_predictions, ignore_index=True)
    print(f"\nTotal KumoRFM predictions: {len(kumo_results_df)}")
else:
    print("No predictions returned. Check API key and graph setup.")
    kumo_results_df = None

In [ ]:
# ==============================================================
# 5.2 Evaluate KumoRFM predictions
# ==============================================================

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve
)

if kumo_results_df is not None:
    eval_df = ground_truth.merge(kumo_results_df, on="buyer_id", how="inner")
    pred_cols = [c for c in eval_df.columns
                 if c not in ["buyer_id", "is_defaulter"]]
    pred_col = pred_cols[0] if pred_cols else None

    if pred_col:
        y_true = eval_df["is_defaulter"].values
        y_score = eval_df[pred_col].values

        roc_auc = roc_auc_score(y_true, y_score)
        pr_auc = average_precision_score(y_true, y_score)

        prec_arr, rec_arr, thresholds = precision_recall_curve(y_true, y_score)
        f1_at_95 = 0.0
        for p, r, t in zip(prec_arr, rec_arr, thresholds):
            if p >= 0.95 and r > 0:
                f1 = 2 * p * r / (p + r)
                f1_at_95 = max(f1_at_95, f1)

        print("=" * 50)
        print("KumoRFM Default Detection Results")
        print("=" * 50)
        print(f"ROC-AUC:          {roc_auc:.4f}")
        print(f"PR-AUC:           {pr_auc:.4f}")
        print(f"F1 @ 95% Prec:    {f1_at_95:.4f}")

        kumo_metrics = {"ROC-AUC": roc_auc, "PR-AUC": pr_auc,
                        "F1@95%Prec": f1_at_95}
    else:
        print("Could not identify prediction column.")
        kumo_metrics = None
else:
    print("Skipping evaluation (no KumoRFM predictions).")
    kumo_metrics = None

## 6. XGBoost Baseline with Hand-Engineered Trade Credit Features

We build a strong XGBoost model using features a production credit risk team would engineer:

**Invoice-level features:** Total outstanding amount, average invoice size, terms distribution
**Payment behavior features:** Average days-past-due, payment-to-invoice ratio, late payment frequency
**Concentration features:** Number of active suppliers, supplier diversity (HHI), top-supplier exposure share
**Bureau features:** Bureau score, years in business
**Velocity features:** Invoice count and total amount in recent windows

In [ ]:
# ==============================================================
# 6.1 Engineer flat features
# ==============================================================

import xgboost as xgb

def build_credit_features(buyers, invoices, payments, credit_lines, bs_pairs):
    features = buyers[["buyer_id"]].copy()

    inv_agg = invoices.groupby("buyer_id").agg(
        total_invoices=("invoice_id", "count"),
        total_invoice_amount=("amount", "sum"),
        avg_invoice_amount=("amount", "mean"),
        max_invoice_amount=("amount", "max"),
        n_suppliers_used=("supplier_id", "nunique"),
        avg_terms=("terms_days", "mean"),
    ).reset_index()

    latest = invoices["invoice_date"].max()
    recent_inv = invoices[invoices["invoice_date"] >= latest - pd.Timedelta(days=30)]
    recent_agg = recent_inv.groupby("buyer_id").agg(
        recent_invoice_count=("invoice_id", "count"),
        recent_invoice_sum=("amount", "sum"),
    ).reset_index()

    pay_inv = payments.merge(
        invoices[["invoice_id", "due_date", "amount"]],
        on="invoice_id", how="left"
    )
    pay_inv["dpd"] = (pay_inv["payment_date"] - pay_inv["due_date"]).dt.days
    pay_inv["pay_ratio"] = pay_inv["payment_amount"] / pay_inv["amount"].clip(lower=1)

    pay_agg = pay_inv.groupby("buyer_id").agg(
        avg_dpd=("dpd", "mean"),
        max_dpd=("dpd", "max"),
        std_dpd=("dpd", "std"),
        avg_pay_ratio=("pay_ratio", "mean"),
        n_late_payments=("dpd", lambda x: (x > 0).sum()),
        n_payments=("payment_id", "count"),
    ).reset_index()
    pay_agg["late_rate"] = pay_agg["n_late_payments"] / pay_agg["n_payments"].clip(lower=1)

    sup_exposure = invoices.groupby(["buyer_id", "supplier_id"])["amount"].sum().reset_index()
    buyer_total = sup_exposure.groupby("buyer_id")["amount"].sum().reset_index()
    buyer_total.columns = ["buyer_id", "total_amount"]
    sup_exposure = sup_exposure.merge(buyer_total, on="buyer_id")
    sup_exposure["share"] = sup_exposure["amount"] / sup_exposure["total_amount"].clip(lower=1)
    sup_exposure["share_sq"] = sup_exposure["share"] ** 2
    hhi = sup_exposure.groupby("buyer_id")["share_sq"].sum().reset_index()
    hhi.columns = ["buyer_id", "supplier_hhi"]
    top_share = sup_exposure.groupby("buyer_id")["share"].max().reset_index()
    top_share.columns = ["buyer_id", "top_supplier_share"]

    total_limit = credit_lines.groupby("buyer_id")["credit_limit"].sum().reset_index()
    total_limit.columns = ["buyer_id", "total_credit_limit"]

    features = features.merge(inv_agg, on="buyer_id", how="left")
    features = features.merge(recent_agg, on="buyer_id", how="left")
    features = features.merge(pay_agg, on="buyer_id", how="left")
    features = features.merge(hhi, on="buyer_id", how="left")
    features = features.merge(top_share, on="buyer_id", how="left")
    features = features.merge(total_limit, on="buyer_id", how="left")
    features = features.merge(
        buyers[["buyer_id", "bureau_score", "years_in_business", "buyer_size"]],
        on="buyer_id", how="left"
    )

    features["utilization"] = (
        features["total_invoice_amount"] /
        features["total_credit_limit"].clip(lower=1)
    )
    features["buyer_size"] = features["buyer_size"].map(
        {"micro": 0, "small": 1, "medium": 2}
    ).fillna(0)

    features = features.fillna(0)
    return features

flat_features = build_credit_features(
    buyers_df, invoices_df, payments_df, credit_lines_df, bs_pairs_df
)
feature_cols = [c for c in flat_features.columns if c != "buyer_id"]

print(f"Feature matrix: {flat_features.shape}")
print(f"Features: {feature_cols}")

In [ ]:
# ==============================================================
# 6.2 Train and evaluate XGBoost
# ==============================================================

train_mask = ~flat_features["buyer_id"].isin(test_buyer_ids)
test_mask = flat_features["buyer_id"].isin(test_buyer_ids)

X_train = flat_features.loc[train_mask, feature_cols].values
y_train = buyers_df.loc[
    buyers_df["buyer_id"].isin(flat_features.loc[train_mask, "buyer_id"])
]["is_defaulter"].values

X_test = flat_features.loc[test_mask, feature_cols].values
y_test = ground_truth.set_index("buyer_id").loc[
    flat_features.loc[test_mask, "buyer_id"]
]["is_defaulter"].values

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    eval_metric="aucpr",
    random_state=42,
    use_label_encoder=False,
)
xgb_model.fit(X_train, y_train, verbose=False)

y_score_xgb = xgb_model.predict_proba(X_test)[:, 1]

roc_auc_xgb = roc_auc_score(y_test, y_score_xgb)
pr_auc_xgb = average_precision_score(y_test, y_score_xgb)

prec_x, rec_x, thresh_x = precision_recall_curve(y_test, y_score_xgb)
f1_at_95_xgb = 0.0
for p, r, t in zip(prec_x, rec_x, thresh_x):
    if p >= 0.95 and r > 0:
        f1 = 2 * p * r / (p + r)
        f1_at_95_xgb = max(f1_at_95_xgb, f1)

print("=" * 50)
print("XGBoost Default Detection Results")
print("=" * 50)
print(f"ROC-AUC:          {roc_auc_xgb:.4f}")
print(f"PR-AUC:           {pr_auc_xgb:.4f}")
print(f"F1 @ 95% Prec:    {f1_at_95_xgb:.4f}")

xgb_metrics = {"ROC-AUC": roc_auc_xgb, "PR-AUC": pr_auc_xgb,
               "F1@95%Prec": f1_at_95_xgb}

## 7. Head-to-Head Comparison: KumoRFM vs XGBoost

In [ ]:
if kumo_metrics:
    comparison_data = {
        "Metric": ["ROC-AUC", "PR-AUC", "F1 @ 95% Precision"],
        "KumoRFM": [kumo_metrics["ROC-AUC"], kumo_metrics["PR-AUC"],
                     kumo_metrics["F1@95%Prec"]],
        "XGBoost": [xgb_metrics["ROC-AUC"], xgb_metrics["PR-AUC"],
                     xgb_metrics["F1@95%Prec"]],
    }
    comp_df = pd.DataFrame(comparison_data)
    comp_df["Delta"] = comp_df["KumoRFM"] - comp_df["XGBoost"]
    comp_df["Delta%"] = (comp_df["Delta"] / comp_df["XGBoost"] * 100).round(1)
    print(comp_df.to_string(index=False))
    print()

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(comparison_data["Metric"]))
    width = 0.3
    bars1 = ax.bar(x - width/2, comparison_data["KumoRFM"], width,
                   label="KumoRFM", color="#FC1373", alpha=0.85)
    bars2 = ax.bar(x + width/2, comparison_data["XGBoost"], width,
                   label="XGBoost", color="steelblue", alpha=0.85)
    ax.set_ylabel("Score")
    ax.set_title("Trade Credit Default Detection: KumoRFM vs XGBoost")
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_data["Metric"])
    ax.legend()
    ax.set_ylim(0, 1.1)
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("KumoRFM metrics not available. XGBoost results only:")
    for k, v in xgb_metrics.items():
        print(f"  {k}: {v:.4f}")
    print("\nRe-run with a valid KumoRFM API key for the full comparison.")

## 8. Portfolio-Level Analysis: Approval Rate vs Expected Loss

In production underwriting, the key business decision is where to set the risk threshold. A lower threshold approves more buyers but increases expected losses. We plot this tradeoff to show how better model discrimination translates into business value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

thresholds_range = np.linspace(0.01, 0.99, 100)
xgb_approval = []
xgb_default_rate = []

for t in thresholds_range:
    approved = y_score_xgb <= t
    if approved.sum() > 0:
        xgb_approval.append(approved.mean())
        xgb_default_rate.append(y_test[approved].mean())
    else:
        xgb_approval.append(0)
        xgb_default_rate.append(0)

axes[0].plot(xgb_approval, xgb_default_rate,
             label="XGBoost", color="steelblue", linewidth=2)

if kumo_metrics and kumo_results_df is not None:
    kumo_approval = []
    kumo_default_rate = []
    kumo_y = eval_df["is_defaulter"].values
    kumo_s = eval_df[pred_col].values

    for t in thresholds_range:
        approved = kumo_s <= t
        if approved.sum() > 0:
            kumo_approval.append(approved.mean())
            kumo_default_rate.append(kumo_y[approved].mean())
        else:
            kumo_approval.append(0)
            kumo_default_rate.append(0)

    axes[0].plot(kumo_approval, kumo_default_rate,
                 label="KumoRFM", color="#FC1373", linewidth=2)

axes[0].set_xlabel("Approval Rate")
axes[0].set_ylabel("Default Rate (among approved)")
axes[0].set_title("Approval Rate vs Default Rate")
axes[0].legend()
axes[0].axhline(y=y_test.mean(), color="gray", linestyle="--",
                alpha=0.5, label="Population default rate")

importances = xgb_model.feature_importances_
feat_imp = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values("Importance", ascending=True).tail(12)

axes[1].barh(feat_imp["Feature"], feat_imp["Importance"],
             color="steelblue", alpha=0.8)
axes[1].set_xlabel("Feature Importance (Gain)")
axes[1].set_title("XGBoost: Top 12 Features")

plt.tight_layout()
plt.show()

print("Note: KumoRFM captures supplier-network and cross-buyer contagion")
print("signals that these flat features cannot represent.")

## 9. Supply-Chain Risk Graph Visualization

Visualizing the buyer-supplier network around default clusters reveals the structural patterns that KumoRFM captures. In production, these subgraphs help credit analysts understand *why* a buyer was flagged and assess portfolio-level concentration risk.

In [ ]:
import networkx as nx

G = nx.Graph()

viz_default_buyers = list(default_buyers)[:20]
viz_suppliers = set()

for bid in viz_default_buyers:
    G.add_node(bid, node_type="buyer", color="crimson")
    buyer_sups = bs_pairs_df[bs_pairs_df["buyer_id"] == bid]["supplier_id"].tolist()
    for sid in buyer_sups:
        viz_suppliers.add(sid)
        G.add_node(sid, node_type="supplier",
                   color="orange" if sid == risky_supplier else "lightgreen")
        n_defaults = invoices_df[
            (invoices_df["buyer_id"] == bid) &
            (invoices_df["supplier_id"] == sid) &
            (invoices_df["is_defaulted"])
        ].shape[0]
        G.add_edge(bid, sid, weight=max(n_defaults, 1),
                   color="red" if n_defaults > 0 else "lightgray")

clean_connected = []
for sid in list(viz_suppliers)[:5]:
    connected = bs_pairs_df[
        (bs_pairs_df["supplier_id"] == sid) &
        (~bs_pairs_df["buyer_id"].isin(default_buyers))
    ]["buyer_id"].head(3).tolist()
    for bid in connected:
        if bid not in G.nodes():
            G.add_node(bid, node_type="buyer", color="lightblue")
            G.add_edge(bid, sid, weight=1, color="lightgray")
            clean_connected.append(bid)

fig, ax = plt.subplots(1, 1, figsize=(15, 10))
pos = nx.spring_layout(G, seed=42, k=2.0, iterations=60)

node_colors = [G.nodes[n].get("color", "gray") for n in G.nodes()]
edge_colors = [G.edges[e].get("color", "gray") for e in G.edges()]
edge_widths = [G.edges[e].get("weight", 1) * 0.8 for e in G.edges()]
node_sizes = [600 if G.nodes[n].get("node_type") == "supplier" else 300
              for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.85, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors,
                       width=edge_widths, alpha=0.5, ax=ax)

labels = {}
for n in G.nodes():
    if G.nodes[n].get("node_type") == "supplier":
        labels[n] = n
    elif G.nodes[n].get("color") == "crimson":
        labels[n] = n
nx.draw_networkx_labels(G, pos, labels, font_size=6, ax=ax)

ax.set_title("Supply-Chain Risk Graph: Buyer-Supplier Default Contagion",
             fontsize=13, fontweight="bold")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="crimson", label="Default Buyer"),
    Patch(facecolor="lightblue", label="Performing Buyer"),
    Patch(facecolor="orange", label=f"Risky Supplier ({risky_supplier})"),
    Patch(facecolor="lightgreen", label="Other Supplier"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=9)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Subgraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Default buyers shown: {len(viz_default_buyers)}")
print(f"Risky supplier ({risky_supplier}) connects to "
      f"{len(concentration_buyers)} defaulted buyers in cascade pattern")

## 10. Production Deployment Considerations

### Real-Time Credit Decisioning Architecture

In production, BNPL and trade credit platforms require **sub-second** underwriting decisions at checkout or invoice creation:

```
[New Invoice / BNPL Request]
         |
         v
   [Identity & Eligibility]  --> KYC, sanctions screening
         |
         v
   [KumoRFM Risk Score]      --> Graph-aware default probability
         |
         v
   [Credit Policy Engine]    --> Limit check, terms assignment
         |                       (risk-based pricing)
         v
   [Decision: Approve / Decline / Refer]
         |
         v
   [Portfolio Monitor]       --> Concentration alerts,
                                 contagion early warning
```

### Key Production Considerations

**1. Real-Time Limit Setting**

Embedded finance platforms need instant credit decisions. KumoRFM scores can feed directly into a limit-setting formula: `approved_limit = base_limit * (1 - risk_score)`. The graph context (supplier health, buyer payment trajectory) produces more accurate limits than bureau-only approaches.

**2. Supplier Concentration Monitoring**

Production systems should continuously monitor supplier-level exposure. When a supplier's buyer portfolio deteriorates (rising average DPD, increasing default rate), all buyers with exposure to that supplier should be re-scored. KumoRFM captures this naturally through its graph structure.

**3. Regulatory Requirements**

Trade credit and BNPL products face increasing regulation (EU Consumer Credit Directive, CFPB BNPL guidance, RBI digital lending guidelines). Key requirements include affordability assessment documentation, fair lending compliance, and transparent adverse action notices. KumoRFM's prediction explanations support these requirements.

**4. Portfolio-Level Risk Management**

Beyond individual buyer scoring, platforms need portfolio-level views: total exposure by industry, geographic concentration, supplier dependency graphs, and expected loss forecasts. The relational graph that feeds KumoRFM is the same data structure needed for these portfolio analytics.

## 11. Summary

This notebook demonstrated BNPL and trade credit default prediction using KumoRFM's relational graph approach across three distinct risk patterns (gradual deterioration, supplier concentration cascade, and bust-out).

**Key takeaways:**

1. **Trade credit risk is inherently relational**: Default risk propagates through buyer-supplier networks. A buyer's risk depends not just on their own payment history, but on the health of their supplier relationships and the behavior of other buyers in the same network.

2. **Concentration risk requires graph structure**: Supplier concentration cascade (Pattern 2) is invisible to flat features because it requires reasoning about which other buyers share the same suppliers. KumoRFM captures this natively through multi-hop graph traversal.

3. **No feature engineering required**: The XGBoost baseline needed 20+ hand-engineered features (DPD velocity, HHI concentration, payment ratios, utilization). KumoRFM operates directly on the five relational tables without manual feature construction.

4. **Portfolio-level business value**: Better risk discrimination translates directly into higher approval rates at the same loss level, or lower losses at the same approval rate. The approval-rate vs expected-loss curve (Section 8) quantifies this business impact.

5. **Natural extension of fraud detection**: This notebook extends the BFSI series from fraud detection (Modules 1-4) into credit risk underwriting, demonstrating that the same relational graph infrastructure serves both use cases.

### Next in the series

- **Module 6**: SME Business Loan Default Prediction (multi-entity graphs: business, owners, invoices, suppliers, bank transactions)

---

*Built by [Abhinav Khare](https://github.com/AbhinavKhareTech) as a community contribution to [kumo-ai/kumo-rfm](https://github.com/kumo-ai/kumo-rfm).*
*For questions or feedback: khare.abhinav@gmail.com*